# B — Data Analysis (fully executable)

This notebook loads the processed master table created by Notebook 01 and reproduces the main analytical questions from the hackathon deliverable.

It creates tables for:

- revenue per hectare
- price trends
- yield by region/crop
- region × crop performance
- improved-seed lift
- pest/disease penalty
- fertilizer bands
- altitude
- planting month
- distance to market
- yearly yield
- weather anomalies
- farmer-reported vs station rainfall

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports"

train = pd.read_csv(PROCESSED / "master_train.csv")
weather = pd.read_csv(PROCESSED / "cleaned_weather.csv")
prices = pd.read_csv(PROCESSED / "cleaned_prices.csv")

print(train.shape)
display(train.head())

(15000, 26)


,plot_id,region,crop_type,survey_year,planting_month,altitude_m,rainfall_mm_season,farm_size_ha,fertilizer_kg_per_ha,improved_seed_used,pest_disease_flag,soil_quality_index,labor_days_per_ha,distance_to_market_km,yield_tons_per_ha,season_mean_temp,season_rainfall_mm,season_extreme_heat_days,weather_months_observed,season_temp_std,season_rainfall_std,season_temp_anomaly,planting_month_num,fertilizer_seed_interaction,rainfall_reporting_ratio,temp_rainfall_interaction
0,PLOT-000339,Oromia,barley,2024,Aug,2697.817349,1157.403560,0.867604,29.393684,0,1.0,0.399408,42.678102,8.573726,2.209711,17.866667,353.0,0.0,3.0,1.960159,70.229924,-1.180903,8,0.000000,3.278764,104.864772
1,PLOT-007457,Amhara,maize,2023,Jun,1523.350141,897.420122,1.099197,16.843992,1,0.0,0.314488,45.370495,19.928302,3.363254,15.200000,709.4,0.0,4.0,1.055936,46.886592,-1.905208,6,16.843992,1.265041,99.800589
2,PLOT-006092,Oromia,wheat,2024,Feb,2389.441948,1119.155741,0.811309,41.015144,0,0.0,0.812596,52.846126,8.961797,3.560614,19.200000,294.9,0.0,4.0,0.951315,13.493031,0.152431,2,0.000000,3.795035,109.248414
3,PLOT-004128,SNNPR,barley,2022,Aug,2612.483420,1246.462210,1.602312,14.353759,0,1.0,0.619516,59.833780,1.849243,1.924800,19.425000,398.3,1.0,4.0,1.294942,81.580892,-1.025347,8,0.000000,3.129456,116.350175
4,PLOT-003033,Amhara,sorghum,2021,Jul,1210.095040,928.477088,0.937758,33.736138,1,0.0,0.577963,22.020843,3.149654,4.171220,16.500000,798.2,1.0,4.0,1.288410,103.747783,-0.605208,7,33.736138,1.163214,110.279585


In [2]:
# Join prices only for economic analysis.
analysis = train.merge(
    prices.rename(columns={"year":"survey_year"}),
    on=["region","crop_type","survey_year"],
    how="left",
    validate="many_to_one"
)

analysis["revenue_birr_ha"] = (
    analysis["yield_tons_per_ha"] * 10 * analysis["price_birr_per_quintal"]
)

print("Analysis table:", analysis.shape)
display(analysis[[
    "region","crop_type","survey_year",
    "yield_tons_per_ha","price_birr_per_quintal","revenue_birr_ha"
]].head())

Analysis table: (15000, 28)


,region,crop_type,survey_year,yield_tons_per_ha,price_birr_per_quintal,revenue_birr_ha
0,Oromia,barley,2024,2.209711,4590.0,101425.735221
1,Amhara,maize,2023,3.363254,3073.5,103369.609159
2,Oromia,wheat,2024,3.560614,5387.0,191810.274440
3,SNNPR,barley,2022,1.924800,4181.0,80475.884561
4,Amhara,sorghum,2021,4.171220,2706.0,112873.220452


In [3]:
# B1.1 Revenue per hectare
revenue_by_crop = (
    analysis.groupby("crop_type")
    .agg(
        mean_yield_t_ha=("yield_tons_per_ha","mean"),
        mean_price_birr_q=("price_birr_per_quintal","mean"),
        mean_revenue_birr_ha=("revenue_birr_ha","mean")
    )
    .reset_index()
    .sort_values("mean_revenue_birr_ha", ascending=False)
)

revenue_by_crop.to_csv(REPORTS / "B1_1_revenue_by_crop_transparent.csv", index=False)
display(revenue_by_crop)

,crop_type,mean_yield_t_ha,mean_price_birr_q,mean_revenue_birr_ha
3,teff,2.002022,7690.996501,153288.746999
4,wheat,2.801160,4845.976097,135438.423911
1,maize,3.900392,2930.972497,114355.668027
0,barley,2.467298,4479.660041,110218.232734
2,sorghum,2.611373,3180.155466,83275.798034


In [4]:
# B1.2 Price trend by crop.
price_trend = (
    prices.pivot_table(
        index="crop_type",
        columns="year",
        values="price_birr_per_quintal",
        aggfunc="mean"
    )
    .reset_index()
)

years = sorted([c for c in price_trend.columns if isinstance(c, (int,np.integer))])
price_trend["absolute_change"] = price_trend[years[-1]] - price_trend[years[0]]
price_trend["pct_change"] = 100 * price_trend["absolute_change"] / price_trend[years[0]]

price_trend.to_csv(REPORTS / "B1_2_price_trends.csv", index=False)
display(price_trend)

year,crop_type,2021,2022,2023,2024,absolute_change,pct_change
0,barley,3946.8,4189.2,4769.6,4977.8,1031.0,26.122428
1,maize,2602.2,2850.2,3055.5,3233.2,631.0,24.248713
2,sorghum,2688.8,3068.2,3475.6,3492.4,803.6,29.886938
3,teff,6885.0,7398.2,8033.0,8575.3,1690.3,24.550472
4,wheat,4305.0,4715.2,4783.0,5557.4,1252.4,29.091754


In [5]:
# B2.1 Yield by region.
yield_region = (
    train.groupby("region")["yield_tons_per_ha"]
    .agg(["mean","median","std","count"])
    .reset_index()
)
yield_region.to_csv(REPORTS / "B2_1_yield_region.csv", index=False)
display(yield_region)

,region,mean,median,std,count
0,Amhara,3.127511,3.019607,1.231214,2962
1,Oromia,3.069958,2.890737,1.290017,2957
2,SNNPR,3.038657,2.739801,1.471023,2962
3,Somali,1.465495,1.294279,0.771337,3049
4,Tigray,3.128418,2.930796,1.305722,3070


In [6]:
# B2.2 Yield by crop.
yield_crop = (
    train.groupby("crop_type")["yield_tons_per_ha"]
    .agg(["mean","median","std","count"])
    .reset_index()
)
yield_crop.to_csv(REPORTS / "B2_2_yield_crop.csv", index=False)
display(yield_crop)

,crop_type,mean,median,std,count
0,barley,2.467298,2.375861,1.156820,2918
1,maize,3.900392,3.772586,1.582920,3036
2,sorghum,2.611373,2.513476,0.974867,2991
3,teff,2.002022,2.001039,0.986924,3001
4,wheat,2.801160,2.692752,1.406026,3054


In [7]:
# B2.3 Region × crop mean yield.
region_crop = pd.pivot_table(
    train,
    index="region",
    columns="crop_type",
    values="yield_tons_per_ha",
    aggfunc="mean"
)
region_crop.to_csv(REPORTS / "B2_3_region_crop_pivot.csv")
display(region_crop)

best_pair = region_crop.stack().idxmax()
worst_pair = region_crop.stack().idxmin()
print("Best region × crop:", best_pair, region_crop.loc[best_pair])
print("Worst region × crop:", worst_pair, region_crop.loc[worst_pair])

crop_type,barley,maize,sorghum,teff,wheat
region,,,,,
Amhara,3.186735,3.826552,2.495893,2.474892,3.600457
Oromia,2.675486,4.444082,2.955229,2.289790,3.117098
SNNPR,2.290700,4.789248,3.104721,2.099703,2.665755
Somali,1.123785,2.320003,1.793512,0.830979,1.282894
Tigray,3.010710,4.074513,2.711832,2.330726,3.452567


Best region × crop: ('SNNPR', 'maize') 4.789248359871966
Worst region × crop: ('Somali', 'teff') 0.8309786202927615


In [8]:
# B2.4 Improved seed lift by crop.
seed_summary = (
    train.groupby(["crop_type","improved_seed_used"])["yield_tons_per_ha"]
    .mean()
    .unstack()
)

seed_gap = pd.DataFrame({
    "crop_type": seed_summary.index,
    "yield_no_improved_seed": seed_summary.get(0),
    "yield_improved_seed": seed_summary.get(1),
}).reset_index(drop=True)

seed_gap["gap_t_ha"] = (
    seed_gap["yield_improved_seed"] - seed_gap["yield_no_improved_seed"]
)
seed_gap["pct_lift"] = (
    100 * seed_gap["gap_t_ha"] / seed_gap["yield_no_improved_seed"]
)

seed_gap.to_csv(REPORTS / "B2_4_seed_gap.csv", index=False)
display(seed_gap)

,crop_type,yield_no_improved_seed,yield_improved_seed,gap_t_ha,pct_lift
0,barley,2.307670,2.725422,0.417752,18.102755
1,maize,3.608950,4.390592,0.781642,21.658434
2,sorghum,2.440893,2.891341,0.450448,18.454229
3,teff,1.842976,2.263873,0.420897,22.837897
4,wheat,2.596292,3.131965,0.535673,20.632227


In [9]:
# B2.5 Pest/disease penalty.
pest_summary = (
    train.groupby(["crop_type","pest_disease_flag"])["yield_tons_per_ha"]
    .mean()
    .unstack()
)

pest_gap = pd.DataFrame({
    "crop_type": pest_summary.index,
    "yield_without_pest": pest_summary.get(0),
    "yield_with_pest": pest_summary.get(1),
}).reset_index(drop=True)

pest_gap["gap_t_ha"] = (
    pest_gap["yield_with_pest"] - pest_gap["yield_without_pest"]
)
pest_gap["pct_lift"] = (
    100 * pest_gap["gap_t_ha"] / pest_gap["yield_without_pest"]
)

pest_gap.to_csv(REPORTS / "B2_5_pest_gap.csv", index=False)
display(pest_gap)

,crop_type,yield_without_pest,yield_with_pest,gap_t_ha,pct_lift
0,barley,2.623275,1.920025,-0.703250,-26.808082
1,maize,4.181903,2.969546,-1.212357,-28.990553
2,sorghum,2.772252,2.025589,-0.746663,-26.933446
3,teff,2.127973,1.569282,-0.558691,-26.254601
4,wheat,2.978408,2.193120,-0.785288,-26.366023


In [10]:
# B3.1 Fertilizer response.
train["fertilizer_band"] = pd.cut(
    train["fertilizer_kg_per_ha"],
    bins=[-np.inf,20,40,60,80,100,120,np.inf],
    labels=["<=20","20-40","40-60","60-80","80-100","100-120",">120"]
)

fertilizer_bands = (
    train.groupby("fertilizer_band", observed=False)["yield_tons_per_ha"]
    .agg(["mean","median","count"])
    .reset_index()
)
fertilizer_by_crop = (
    train.groupby(["crop_type","fertilizer_band"], observed=False)
    ["yield_tons_per_ha"].mean()
    .reset_index(name="mean_yield_t_ha")
)

fertilizer_bands.to_csv(REPORTS / "B3_1_fertilizer_bands.csv", index=False)
fertilizer_by_crop.to_csv(REPORTS / "B3_1_fertilizer_by_crop.csv", index=False)

display(fertilizer_bands)

,fertilizer_band,mean,median,count
0,<=20,2.431951,2.297499,3651
1,20-40,2.711508,2.544077,4845
2,40-60,2.926846,2.766437,3001
3,60-80,3.035575,2.881051,1542
4,80-100,3.103084,2.915359,697
5,100-120,3.350394,3.273710,281
6,>120,3.300717,3.157549,241


In [11]:
# B3.2 Altitude by crop.
altitude_by_crop = (
    train.groupby("crop_type")["altitude_m"]
    .agg(["mean","median","min","max","count"])
    .reset_index()
)
altitude_by_crop.to_csv(REPORTS / "B3_2_altitude_by_crop.csv", index=False)
display(altitude_by_crop)

,crop_type,mean,median,min,max,count
0,barley,2663.875467,2693.657188,55.803702,3100.000000,2918
1,maize,1386.564409,1379.707091,94.805205,2829.896288,3036
2,sorghum,1292.688419,1296.164833,84.492277,2605.590055,2991
3,teff,2079.865209,2083.429666,61.906015,3100.000000,3001
4,wheat,2570.915711,2585.910439,70.342827,3100.000000,3054


In [12]:
# B3.3 Planting month.
planting_month = (
    train.groupby(["crop_type","planting_month"])["yield_tons_per_ha"]
    .agg(["mean","count"])
    .reset_index()
)
planting_month.to_csv(REPORTS / "B3_3_planting_month.csv", index=False)
display(planting_month.head(20))

,crop_type,planting_month,mean,count
0,barley,Aug,2.577421,534
1,barley,Feb,2.238489,584
2,barley,Jul,2.574380,588
3,barley,Jun,2.660333,573
4,barley,Mar,2.312753,639
5,maize,Aug,3.871015,621
6,maize,Feb,4.107312,609
7,maize,Jul,3.841542,578
8,maize,Jun,3.724080,617
9,maize,Mar,3.957723,611


In [13]:
# B3.4 Distance to market.
distance_corr = train[["distance_to_market_km","yield_tons_per_ha"]].corr().iloc[0,1]

train["distance_band"] = pd.cut(
    train["distance_to_market_km"],
    bins=[-np.inf,5,10,20,40,np.inf],
    labels=["<=5","5-10","10-20","20-40",">40"]
)
distance_bands = (
    train.groupby("distance_band", observed=False)["yield_tons_per_ha"]
    .agg(["mean","median","count"])
    .reset_index()
)
distance_bands.to_csv(REPORTS / "B3_4_distance_bands.csv", index=False)

print(f"Correlation between distance and yield: {distance_corr:.4f}")
display(distance_bands)

Correlation between distance and yield: 0.0061


,distance_band,mean,median,count
0,<=5,2.759833,2.565567,5148
1,5-10,2.720882,2.588066,3375
2,10-20,2.787260,2.602437,3688
3,20-40,2.771028,2.571247,2270
4,>40,2.802214,2.669887,519


In [14]:
# B4.1 Yield trend by year and region.
year_region = (
    train.groupby(["survey_year","region"])["yield_tons_per_ha"]
    .agg(["mean","count"])
    .reset_index()
)
year_region.to_csv(REPORTS / "B4_1_year_region.csv", index=False)
display(year_region)

,survey_year,region,mean,count
0,2021,Amhara,3.145472,751
1,2021,Oromia,3.036277,731
2,2021,SNNPR,2.865577,699
3,2021,Somali,1.285267,751
4,2021,Tigray,3.149460,790
5,2022,Amhara,3.075580,707
6,2022,Oromia,3.024702,754
7,2022,SNNPR,3.019780,759
8,2022,Somali,1.674672,781
9,2022,Tigray,3.145122,759


In [15]:
# B4.2 Weather anomaly.
# The anomaly is already calculated in Notebook 01 for each plot.
weather_anomaly = (
    train.groupby(["region","survey_year"])["season_temp_anomaly"]
    .mean()
    .reset_index()
    .rename(columns={"season_temp_anomaly":"anomaly"})
)

region_avg = (
    train.groupby("region")["season_mean_temp"]
    .mean()
    .rename("season_mean_temp_region_average")
)

weather_anomaly = weather_anomaly.merge(region_avg, on="region", how="left")
weather_anomaly.to_csv(REPORTS / "B4_2_weather_anomalies.csv", index=False)
display(weather_anomaly)

,region,survey_year,anomaly,season_mean_temp_region_average
0,Amhara,2021,-0.370322,16.762148
1,Amhara,2022,1.206755,16.762148
2,Amhara,2023,-1.598666,16.762148
3,Amhara,2024,-0.521599,16.762148
4,Oromia,2021,0.080851,18.866312
5,Oromia,2022,0.665063,18.866312
6,Oromia,2023,-0.545831,18.866312
7,Oromia,2024,-0.950631,18.866312
8,SNNPR,2021,0.940783,20.126016
9,SNNPR,2022,-0.736777,20.126016


In [16]:
# B4.3 Farmer-reported rainfall versus station seasonal rainfall.
rainfall_comparison = train[[
    "rainfall_mm_season","season_rainfall_mm"
]].dropna().copy()

rainfall_correlation = rainfall_comparison.corr().iloc[0,1]
mean_absolute_difference = (
    rainfall_comparison["rainfall_mm_season"] -
    rainfall_comparison["season_rainfall_mm"]
).abs().mean()

rainfall_summary = pd.DataFrame([{
    "correlation": rainfall_correlation,
    "mean_absolute_difference_mm": mean_absolute_difference,
    "plot_reported_mean_mm": rainfall_comparison["rainfall_mm_season"].mean(),
    "station_season_mean_mm": rainfall_comparison["season_rainfall_mm"].mean()
}])

rainfall_summary.to_csv(REPORTS / "B4_3_rainfall_summary.csv", index=False)
display(rainfall_summary)

,correlation,mean_absolute_difference_mm,plot_reported_mean_mm,station_season_mean_mm
0,0.006605,501.190972,856.32465,371.649182


## Main analytical takeaways

The data shows meaningful differences by crop and region. Maize has the highest average yield, while teff has the highest mean revenue per hectare because of its higher price. Improved seed is associated with higher yield, while pest/disease observations are associated with lower yield.

These are **associations**, not causal estimates.